# SynID Demo

Zero-shot identity-consistent image generation from text alone.  
No real reference photos. No dataset. No pretraining. ~5 min on T4 GPU.

**Before running:** Go to `Runtime > Change runtime type > T4 GPU`

In [ ]:
!pip install -q gradio diffusers transformers accelerate controlnet_aux safetensors huggingface_hub insightface onnxruntime torchvision

In [ ]:
!git clone https://github.com/rxbinsingh/SynID
%cd SynID

## Option A — Launch Gradio UI

Click **Load Pipelines** in the UI to download and initialize models (~2 min on first run).  
A public share URL will appear — use it to open the app on your phone.

In [ ]:
exec(open('synid_ui.py').read())

## Option B — Backend scripting

Skip Option A and run these cells instead.

In [ ]:
exec(open('identity_projection_complete.py').read())
init_synid_backend()

In [ ]:
profile = create_character(
    identity_prompt="young woman, curly red hair, green eyes, freckles, photorealistic",
    anchor_seed=42,
    num_identity_tokens=4,
    train_steps=250,
)
display(profile.anchor_image)

In [ ]:
adapters = attach_identity_adapters(pipe.unet, identity_dim=768, scale=0.5)
hooks    = register_adapter_hooks(pipe.unet)

train_adapter_on_bootstrap(
    pipe.unet, pipe.vae, pipe.text_encoder, pipe.tokenizer,
    adapters, profile.identity_tokens,
    [c.image for c in profile.bootstrap_candidates],
    [c.prompt for c in profile.bootstrap_candidates],
    profile.base_identity_embedding,
    train_steps=200, lr=1e-5,
)

In [ ]:
expressions = ["bright smile", "surprised", "serious", "neutral"]
images = []
for i, expr in enumerate(expressions):
    prompt = profile.character_core_prompt + ", " + expr
    img = generate_with_adapter(
        profile.identity_tokens, prompt, profile.pose_image,
        pipe.unet, pipe, seed=5001+i,
        identity_scale=adaptive_identity_scale(prompt),
        generation_steps=30,
    )
    images.append(img)
display(show_grid(images, cols=2))

In [ ]:
exec(open('evaluation_harness.py').read())
results = eval_quick(profile, adapters_active=True)
print(f"CLIP: {results['clip_mean']:.4f}")
arc = results['arc_mean']
print(f"ArcFace: {arc:.4f}" if arc is not None else "ArcFace: N/A")

In [ ]:
save_checkpoint(profile, adapters, "/content/checkpoints/my_character")
print("Saved.")